In [4]:
from __future__ import annotations

import copy
import hashlib
import json
import math
import time
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.utils.prune as prune
import torch.nn.functional as F

from libraries.MLPRegressor import MLPRegressor
from libraries.TrainingState import TrainingState

NameError: name 'SEARCH_LEARNING_RATE' is not defined

## 1. CONFIGURACIÓN

In [5]:
DATASET = "FD001"

RANDOM_STATE = 42

# Cada cuántas épocas se mide la validación durante la búsqueda.
SEARCH_VALIDATION_INTERVAL = 25

# Entrenamiento durante la búsqueda.
SEARCH_LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4

# El early stopping se activa después de este mínimo de épocas para no detener
# prematuramente redes pequeñas que pueden aprender con mayor lentitud.
FINAL_MIN_EPOCHS_BEFORE_STOP = 500

# Dispositivo. En GPU, las matrices de este MLP son pequeñas, pero el
# entrenamiento de múltiples arquitecturas puede acelerarse.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Archivos de salida.
CACHE_VERSION = 6
OBJECTIVE_NAME = "rmse"
if OBJECTIVE_NAME not in {"rmse", "nasa", "both"}:
    raise ValueError("OBJECTIVE_NAME debe ser 'rmse', 'nasa' o 'both'.")
# En modo both, RMSE elige el checkpoint; MOPSO optimiza RMSE, NASA y parámetros.
CACHE_DIR = Path("mopso_cache")
RESULTS_DIR = Path("mopso_results")
MODELS_DIR = Path("mopso_models")

# Valores de referencia para normalizar RMSE y NASA en el cálculo de un objetivo combinado.
RMSE_REFERENCE = 13.0
NASA_REFERENCE = 3.0

# Configuraciones para la poda
PRUNING_FINE_TUNE_EPOCHS = 1_000
PRUNING_VALIDATION_INTERVAL = 25
PRUNING_EARLY_STOPPING_PATIENCE = 10
PRUNING_MIN_EPOCHS_BEFORE_STOP = 100
PRUNING_ALPHAS = [1, 0.95, 0.90, 0.85, 0.80, 0.75, 0.70, 0.65, 0.60, 0.55, 0.50]

sns.set_theme(style="whitegrid")
plt.rcParams.update({"font.size": 12, "figure.dpi": 180})

## 2. CARGA DE LOS MODELOS

In [6]:
from collections import OrderedDict


def build_candidate_model(candidate: Dict[str, Any]) -> nn.Sequential:
    """Reconstruye un candidato JSON como una red densa torch.nn."""
    representative = candidate.get("representative_model")
    if not isinstance(representative, dict):
        raise ValueError(
            f"El candidato {candidate.get('candidate_id')} "
            "no tiene un representative_model válido."
        )

    layer_configs = representative.get("layers")
    if not isinstance(layer_configs, list) or not layer_configs:
        raise ValueError(
            f"El candidato {candidate.get('candidate_id')} no tiene capas."
        )

    activation_factories = {
        "linear": nn.Identity,
        "identity": nn.Identity,
        "relu": nn.ReLU,
        "elu": nn.ELU,
        "silu": nn.SiLU,
        "tanh": nn.Tanh,
        "sigmoid": nn.Sigmoid,
    }
    modules = OrderedDict()
    previous_size = None

    for layer_index, layer_config in enumerate(layer_configs):
        if not isinstance(layer_config, dict):
            raise ValueError(f"La capa {layer_index} debe ser un objeto JSON.")
        if layer_config.get("type", "dense") != "dense":
            raise ValueError(
                f"Tipo de capa no soportado: {layer_config.get('type')!r}."
            )
        if layer_config.get(
            "weights_layout", "[output_neuron][input_neuron]"
        ) != "[output_neuron][input_neuron]":
            raise ValueError(
                f"Formato de pesos no soportado en la capa {layer_index}."
            )

        weights = torch.as_tensor(
            layer_config.get("weights"),
            dtype=torch.float32,
        )
        bias = torch.as_tensor(layer_config.get("bias"), dtype=torch.float32)
        if weights.ndim != 2:
            raise ValueError(f"Los pesos de la capa {layer_index} deben ser 2D.")
        output_size, input_size = weights.shape
        if (
            input_size != int(layer_config.get("input_size", -1))
            or output_size != int(layer_config.get("output_size", -1))
        ):
            raise ValueError(
                f"Las dimensiones declaradas no coinciden con los pesos "
                f"de la capa {layer_index}."
            )
        if previous_size is not None and input_size != previous_size:
            raise ValueError(
                f"La capa {layer_index} espera {input_size} entradas, "
                f"pero la capa anterior produce {previous_size}."
            )
        if bias.shape != (output_size,):
            raise ValueError(
                f"El bias de la capa {layer_index} debe tener "
                f"dimensión ({output_size},)."
            )
        if not torch.isfinite(weights).all() or not torch.isfinite(bias).all():
            raise ValueError(f"La capa {layer_index} contiene valores no finitos.")

        linear = nn.Linear(input_size, output_size, bias=True)
        with torch.no_grad():
            linear.weight.copy_(weights)
            linear.bias.copy_(bias)
        modules[f"linear_{layer_index}"] = linear

        activation_name = str(layer_config.get("activation", "linear")).lower()
        activation_factory = activation_factories.get(activation_name)
        if activation_factory is None:
            raise ValueError(
                f"Activación no soportada en la capa {layer_index}: "
                f"{activation_name}."
            )
        modules[f"activation_{layer_index}"] = activation_factory()

        dropout_probability = float(layer_config.get("dropout_after", 0.0))
        if not 0.0 <= dropout_probability < 1.0:
            raise ValueError(
                f"Dropout inválido en la capa {layer_index}: "
                f"{dropout_probability}."
            )
        if dropout_probability > 0.0:
            modules[f"dropout_{layer_index}"] = nn.Dropout(
                p=dropout_probability
            )

        previous_size = output_size

    return nn.Sequential(modules).to(DEVICE)


candidates_path = RESULTS_DIR / (
    f"{DATASET}_mopso_final_candidates_v{CACHE_VERSION}_"
    f"{OBJECTIVE_NAME}.json"
)
if not candidates_path.is_file():
    raise FileNotFoundError(f"No se encontró el JSON de candidatos: {candidates_path}")

with candidates_path.open("r", encoding="utf-8") as json_file:
    models_config = json.load(json_file)

if models_config.get("format") != "agnostic-dense-mlp":
    raise ValueError("El archivo no contiene un JSON agnóstico de MLP compatible.")
if int(models_config.get("format_version", -1)) != 1:
    raise ValueError(
        f"Versión de JSON no soportada: {models_config.get('format_version')}."
    )
if models_config.get("dataset") != DATASET:
    raise ValueError(
        f"El JSON pertenece a {models_config.get('dataset')}, "
        f"no a {DATASET}."
    )

candidate_configs = models_config.get("candidates")
if not isinstance(candidate_configs, list) or not candidate_configs:
    raise ValueError("El archivo JSON no contiene candidatos para reconstruir.")

candidate_models = []
for candidate_config in candidate_configs:
    model = build_candidate_model(candidate_config)
    model.eval()
    candidate_models.append(
        {
            "candidate_id": candidate_config["candidate_id"],
            "selected_winner": bool(candidate_config.get("selected_winner", False)),
            "representative_seed": int(
                candidate_config["representative_model"]["seed"]
            ),
            "architecture": candidate_config["architecture"],
            "parameter_count": int(candidate_config["parameter_count"]),
            "test_results": candidate_config.get("test_results", {}),
            "model": model,
        }
    )

print(f"Modelos reconstruidos: {len(candidate_models)}")
for candidate in candidate_models:
    print(
        f"{candidate['candidate_id']} | seed={candidate['representative_seed']} | "
        f"winner={candidate['selected_winner']} | "
        f"architecture={candidate['architecture']['hidden_sizes']}"
    )

Modelos reconstruidos: 4
candidate_01 | seed=44 | winner=False | architecture=[24, 48, 64]
candidate_02 | seed=33 | winner=False | architecture=[4]
candidate_03 | seed=55 | winner=True | architecture=[16, 4, 48]
candidate_04 | seed=44 | winner=False | architecture=[4, 4, 8]


## 4. FUNCIONES DE ENTRENAMIENTO

In [ ]:
def calculate_combined_objective(rmse: float, nasa_mean: float) -> float:
    """Combina RMSE y NASA normalizados con sus valores de referencia."""
    return (
        float(rmse) / RMSE_REFERENCE
        + float(nasa_mean) / NASA_REFERENCE
    ) / 2.0


def calculate_objective(metrics: Dict[str, float]) -> float:
    """Calcula el valor objetivo a partir de las métricas de validación."""
    match OBJECTIVE_NAME:
        case "rmse":
            return float(metrics[OBJECTIVE_NAME])
        case "nasa":
            return float(metrics[OBJECTIVE_NAME])
        case "both":
            return calculate_combined_objective(
                rmse=metrics["rmse"],
                nasa_mean=metrics["nasa_mean"],
            )
        case _:
            raise ValueError(f"Nombre de objetivo no reconocido: {OBJECTIVE_NAME}")


def train_additional_epochs(
    model: MLPRegressor,
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_val: np.ndarray,
    y_val: np.ndarray,
    start_epoch: int,
    additional_epochs: int,
    validation_interval: int,
    previous_best_weights: Optional[Dict[str, torch.Tensor]] = None,
    previous_best_epoch: int = 0,
    previous_best_metrics: Optional[Dict[str, float]] = None,
    early_stopping_patience: Optional[int] = None,
    minimum_epochs_before_stop: int = 0,
    minimum_delta: float = 0.0,
    verbose: bool = False,
) -> TrainingState:
    """Continúa el entrenamiento y conserva dos estados diferentes.

    resume_weights:
        Pesos de la última época. Se utilizan para continuar el entrenamiento.

    best_weights:
        Pesos de la época con menor RMSE o NASA de validación. Se utilizan para medir
        la calidad y para devolver el modelo final.

    Separar ambos estados evita que el early stopping o el mejor checkpoint
    rompan la trayectoria del optimizador utilizada por el warm start.
    """
    if additional_epochs < 0:
        raise ValueError("additional_epochs no puede ser negativo.")
    if validation_interval < 1:
        raise ValueError("validation_interval debe ser positivo.")

    X_train_tensor = torch.as_tensor(
        X_train,
        dtype=torch.float32,
        device=model.device,
    )
    y_train_tensor = torch.as_tensor(
        y_train,
        dtype=torch.float32,
        device=model.device,
    ).reshape(-1, 1)

    if previous_best_metrics is None:
        best_objective = float("inf")
        best_metrics = {
            "mse": float("inf"),
            "rmse": float("inf"),
            "mae": float("inf"),
            "nasa": float("inf"),
            "nasa_mean": float("inf")
        }
        best_weights = model.get_weights()
        best_epoch = int(start_epoch)
    else:
        best_objective = float(calculate_objective(previous_best_metrics))
        best_metrics = copy.deepcopy(previous_best_metrics)
        best_weights = (
            model.get_weights()
            if previous_best_weights is None
            else copy.deepcopy(previous_best_weights)
        )
        best_epoch = int(previous_best_epoch)

    checks_without_improvement = 0
    final_loss = float("nan")
    actual_additional_epochs = 0

    start_time = time.time()

    for local_epoch in range(1, additional_epochs + 1):
        global_epoch = int(start_epoch + local_epoch)
        actual_additional_epochs = local_epoch

        model.model.train()
        model.optimizer.zero_grad(set_to_none=True)

        predictions = model.model(X_train_tensor)
        loss = model.criterion(predictions, y_train_tensor)
        loss.backward()
        model.optimizer.step()
        final_loss = float(loss.item())

        must_validate = (
            local_epoch % validation_interval == 0
            or local_epoch == additional_epochs
        )

        if must_validate:
            validation_predictions = model.predict_clipped(X_val)
            metrics = regression_metrics(y_val, validation_predictions)

            improvement = best_objective - calculate_objective(metrics)
            if improvement > float(minimum_delta):
                best_objective = float(calculate_objective(metrics))
                best_metrics = metrics
                best_weights = model.get_weights()
                best_epoch = global_epoch
                checks_without_improvement = 0
            else:
                checks_without_improvement += 1

            if verbose:
                print(
                    f"      Época {global_epoch} | "
                    f"Loss={final_loss:.4f} | "
                    f"Val {OBJECTIVE_NAME.upper()}="
                    f"{calculate_objective(metrics):.4f} | "
                    f"Mejor {OBJECTIVE_NAME.upper()}="
                    f"{best_objective:.4f} en época {best_epoch}"
                )

            can_stop = (
                early_stopping_patience is not None
                and global_epoch >= minimum_epochs_before_stop
                and checks_without_improvement >= early_stopping_patience
            )
            if can_stop:
                if verbose:
                    print(
                        "      Early stopping: no hubo mejora suficiente "
                        f"durante {checks_without_improvement} controles."
                    )
                break

    elapsed = time.time() - start_time
    total_epochs = int(start_epoch + actual_additional_epochs)

    if verbose:
        print(
            f"    Entrenamiento terminado en {elapsed:.2f}s | "
            f"épocas totales={total_epochs} | mejor época={best_epoch}"
        )

    return TrainingState(
        total_epochs=total_epochs,
        resume_weights=model.get_weights(),
        optimizer_state=model.get_optimizer_state(),
        best_weights=best_weights,
        best_epoch=best_epoch,
        best_metrics=best_metrics,
        final_training_loss=final_loss,
    )


def nasa_score(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """Calcula el NASA Score habitual para C-MAPSS.

    Los errores positivos corresponden a sobreestimación del RUL y reciben
    una penalización más rápida, porque pueden retrasar una intervención.
    """
    errors = np.asarray(y_pred) - np.asarray(y_true)
    errors = np.clip(errors, -200.0, 200.0)

    penalties = np.where(
        errors < 0,
        np.exp(-errors / 13.0) - 1.0,
        np.exp(errors / 10.0) - 1.0,
    )
    return float(np.sum(penalties))


def get_linear_layers(model: MLPRegressor) -> List[nn.Linear]:
    """Devuelve las capas que contienen arcos ponderados."""
    return [
        module
        for module in model.model.modules()
        if isinstance(module, nn.Linear)
    ]


def count_mlp_arcs(model: MLPRegressor) -> int:
    """Número de arcos de la MLP.

    Se cuentan únicamente pesos de capas Linear.
    Los bias no se consideran arcos.
    """
    return int(
        sum(
            layer.weight.numel()
            for layer in get_linear_layers(model)
        )
    )


def count_active_arcs(model: MLPRegressor) -> int:
    """Cuenta los arcos activos durante la poda."""
    active = 0

    for layer in get_linear_layers(model):
        if hasattr(layer, "weight_mask"):
            active += int(
                torch.count_nonzero(layer.weight_mask).item()
            )
        else:
            active += int(layer.weight.numel())

    return active


def test_model(
    model: MLPRegressor,
    dataset_name: str,
    columns_to_drop: Sequence[str],
    selected_features: Sequence[str],
    sensor_scaler: MinMaxScaler,
    derived_scaler: StandardScaler,
    bootstrap_iterations: int = 500,
    confidence_level: float = 0.95,
    model_label: Optional[str] = None,
) -> Dict[str, Any]:
    """Evalúa el test del modelo con intervalos de confianza por bootstrap."""
    label_suffix = f" ({model_label})" if model_label else ""
    print(f"\nProcesando test de {dataset_name}{label_suffix} con bootstrap...")
    start_time = time.time()

    test_path = DATA_DIR / f"test_{dataset_name}.txt"
    rul_path = DATA_DIR / f"RUL_{dataset_name}.txt"

    try:
        test_dataframe = load_cmapss(test_path)
        test_rul_dataframe = load_cmapss_rul(rul_path)
    except FileNotFoundError:
        print(f"No se encontraron archivos de test para {dataset_name}.")
        return {}

    unit_order = test_dataframe["col_1"].drop_duplicates().to_numpy()
    if len(unit_order) != len(test_rul_dataframe):
        raise ValueError(
            "El número de motores de test no coincide con el archivo RUL."
        )

    raw_rul_lookup = pd.Series(
        test_rul_dataframe["rul"].to_numpy(dtype=np.float32),
        index=unit_order,
    )

    test_rul_by_row = compute_rul(
        test_dataframe,
        base_rul=raw_rul_lookup,
        clip_max=RUL_MAX,
    )

    reduced_test = test_dataframe.drop(
        columns=list(columns_to_drop),
        errors="ignore",
    )
    reduced_test = reduced_test[
        ["col_1", "col_2", *selected_features]
    ]

    X_test_raw, _, test_units = build_statistical_windows(
        dataframe=reduced_test,
        rul=test_rul_by_row,
        seq_len=SEQ_LEN,
        sensor_scaler=sensor_scaler,
        statistics=TEMPORAL_STATISTICS,
        only_last=True,
    )
    X_test = derived_scaler.transform(X_test_raw).astype(np.float32)

    y_test_raw = raw_rul_lookup.loc[test_units].to_numpy(dtype=np.float32)
    raw_predictions = model.predict_raw(X_test)

    metrics = regression_metrics(y_test_raw, raw_predictions)
    test_nasa = nasa_score(y_test_raw, raw_predictions)

    if len(y_test_raw) == 0:
        raise ValueError(f"No hay muestras de test para {dataset_name}.")

    bootstrap_rng = np.random.default_rng(RANDOM_STATE)
    n_samples = len(y_test_raw)
    bootstrap_mse = np.empty(bootstrap_iterations, dtype=np.float64)
    bootstrap_rmse = np.empty(bootstrap_iterations, dtype=np.float64)
    bootstrap_mae = np.empty(bootstrap_iterations, dtype=np.float64)
    bootstrap_nasa = np.empty(bootstrap_iterations, dtype=np.float64)

    for index in range(bootstrap_iterations):
        sample_index = bootstrap_rng.integers(
            0,
            n_samples,
            size=n_samples,
        )
        y_boot = y_test_raw[sample_index]
        pred_boot = raw_predictions[sample_index]

        boot_metrics = regression_metrics(y_boot, pred_boot)
        bootstrap_mse[index] = boot_metrics["mse"]
        bootstrap_rmse[index] = boot_metrics["rmse"]
        bootstrap_mae[index] = boot_metrics["mae"]
        bootstrap_nasa[index] = nasa_score(y_boot, pred_boot)

    alpha = (1.0 - confidence_level) / 2.0
    percentiles = [100.0 * alpha, 100.0 * (1.0 - alpha)]

    mse_ci = tuple(
        float(value) for value in np.percentile(bootstrap_mse, percentiles)
    )
    rmse_ci = tuple(
        float(value) for value in np.percentile(bootstrap_rmse, percentiles)
    )
    mae_ci = tuple(
        float(value) for value in np.percentile(bootstrap_mae, percentiles)
    )
    nasa_ci = tuple(
        float(value) for value in np.percentile(bootstrap_nasa, percentiles)
    )

    elapsed = time.time() - start_time
    print(
        f"  RUL: MSE={metrics['mse']:.2f}, "
        f"RMSE={metrics['rmse']:.2f}, "
        f"MAE={metrics['mae']:.2f}, NASA={test_nasa:.2f}"
    )
    print(
        f"  IC {confidence_level * 100:.0f}% por bootstrap "
        f"(B={bootstrap_iterations}): "
        f"MSE=[{mse_ci[0]:.2f}, {mse_ci[1]:.2f}], RMSE=[{rmse_ci[0]:.2f}, {rmse_ci[1]:.2f}], MAE=[{mae_ci[0]:.2f}, {mae_ci[1]:.2f}], NASA=[{nasa_ci[0]:.2f}, {nasa_ci[1]:.2f}]"
    )
    print(f"  Tiempo de test: {elapsed:.2f}s")

    plot_actual_vs_predicted(
        y_true=y_test_raw,
        y_pred=raw_predictions,
        title=f"Actual RUL vs. Predicted RUL: {dataset_name}{label_suffix} (test)",
    )
    plot_test_by_engine(
        engine_ids=test_units,
        y_true=y_test_raw,
        y_pred=raw_predictions,
        dataset_name=dataset_name,
        model_label=model_label,
    )

    return {
        "test_mse": metrics["mse"],
        "test_rmse": metrics["rmse"],
        "test_mae": metrics["mae"],
        "test_nasa": test_nasa,
        "test_raw_target_mse": metrics["mse"],
        "test_mse_ci95": mse_ci,
        "test_rmse_ci95": rmse_ci,
        "test_mae_ci95": mae_ci,
        "test_nasa_ci95": nasa_ci,
        "bootstrap_iterations": int(bootstrap_iterations),
        "confidence_level": float(confidence_level),
    }


def regression_metrics(
    y_true: np.ndarray,
    y_pred: np.ndarray,
) -> Dict[str, float]:
    mse = float(mean_squared_error(y_true, y_pred))
    nasa = float(nasa_score(y_true, y_pred))
    summary = {
        "mse": mse,
        "rmse": float(math.sqrt(mse)),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "nasa": nasa,
        "nasa_mean": float(nasa / len(y_true))
    }
    return summary

## 5. FUNCIONES DE PODA

In [ ]:
def apply_global_magnitude_pruning(
    model: MLPRegressor,
    alpha: float,
) -> Dict[str, int | float]:
    """Aplica poda global respetando A_active <= floor(alpha * A0).

    Parameters
    ----------
    alpha:
        Fracción máxima de arcos que se desea conservar.
        alpha=0.70 implica eliminar al menos el 30 %.
    """
    if not 0.0 < alpha <= 1.0:
        raise ValueError("alpha debe cumplir 0 < alpha <= 1.")

    layers = get_linear_layers(model)
    parameters_to_prune = [
        (layer, "weight")
        for layer in layers
    ]

    total_arcs = count_mlp_arcs(model)
    target_active = int(math.floor(alpha * total_arcs))
    target_pruned = total_arcs - target_active

    if target_pruned > 0:
        # Se usa un entero, en lugar de una fracción, para imponer
        # exactamente el presupuesto deseado de arcos.
        prune.global_unstructured(
            parameters_to_prune,
            pruning_method=prune.L1Unstructured,
            amount=target_pruned,
        )

    active_arcs = count_active_arcs(model)

    return {
        "alpha": float(alpha),
        "total_arcs": int(total_arcs),
        "active_arcs": int(active_arcs),
        "pruned_arcs": int(total_arcs - active_arcs),
        "sparsity": float(1.0 - active_arcs / total_arcs),
    }


def make_pruning_permanent(model: MLPRegressor) -> None:
    """Incorpora las máscaras a los pesos antes de guardar el modelo."""
    for layer in get_linear_layers(model):
        if hasattr(layer, "weight_orig"):
            prune.remove(layer, "weight")


def prune_and_finetune(
    dense_model: MLPRegressor,
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_val: np.ndarray,
    y_val: np.ndarray,
    alpha: float,
) -> Tuple[MLPRegressor, Dict[str, Any]]:
    """Poda una copia del modelo denso y realiza fine-tuning."""

    # Trabajar sobre una copia independiente de la solución MOPSO.
    model = MLPRegressor(
        input_size=dense_model.input_size,
        architecture=dense_model.architecture,
        learning_rate=dense_model.learning_rate,
        weight_decay=dense_model.weight_decay,
        device = DEVICE,

    )
    model.load_weights(dense_model.get_weights())

    pruning_info = apply_global_magnitude_pruning(
        model=model,
        alpha=alpha,
    )

    # Reconstruir AdamW después de crear weight_orig/weight_mask.
    model.optimizer = optim.AdamW(
        model.model.parameters(),
        lr=model.learning_rate,
        weight_decay=model.weight_decay,
    )

    # Evaluar inmediatamente después de podar.
    initial_predictions = model.predict_clipped(X_val)
    initial_metrics = regression_metrics(
        y_val,
        initial_predictions,
    )
    initial_weights = model.get_weights()

    # Fine-tuning manteniendo fija la máscara.
    state = train_additional_epochs(
        model=model,
        X_train=X_train,
        y_train=y_train,
        X_val=X_val,
        y_val=y_val,
        start_epoch=0,
        additional_epochs=PRUNING_FINE_TUNE_EPOCHS,
        validation_interval=PRUNING_VALIDATION_INTERVAL,
        previous_best_weights=initial_weights,
        previous_best_epoch=0,
        previous_best_metrics=initial_metrics,
        early_stopping_patience=PRUNING_EARLY_STOPPING_PATIENCE,
        minimum_epochs_before_stop=PRUNING_MIN_EPOCHS_BEFORE_STOP,
        minimum_delta=0.0,
        verbose=False,
    )

    # Recuperar el mejor checkpoint observado.
    model.load_weights(state.best_weights)

    final_predictions = model.predict_clipped(X_val)
    final_metrics = regression_metrics(
        y_val,
        final_predictions,
    )

    pruning_info.update(
        {
            "rmse_before_finetune": initial_metrics["rmse"],
            "rmse_after_finetune": final_metrics["rmse"],
            "mae_after_finetune": final_metrics["mae"],
            "nasa_after_finetune": nasa_score(
                y_val,
                final_predictions,
            ),
            "best_epoch": state.best_epoch,
        }
    )

    # Convierte weight_orig * weight_mask nuevamente en un weight normal.
    # Los elementos podados quedan exactamente en cero.
    make_pruning_permanent(model)

    return model, pruning_info


def process_pruning(
    processed: Dict[str, Any],
    alpha: float = 0.70,
) -> Dict[str, Any]:
    """Poda una copia del modelo y reporta solamente el test oficial."""
    pruning_start = time.time()
    dataset_name = processed["Dataset"]
    context = processed.get("_pruning_context")
    if context is None:
        raise ValueError(
            "El resultado no contiene '_pruning_context'. "
            "Debe provenir de process_dataset()."
        )
    print(f"\n{'=' * 80}\nPodando {dataset_name}\n{'=' * 80}")

    pruned_model, pruning_results = prune_and_finetune(
        dense_model=context["model"],
        X_train=context["X_train"],
        y_train=context["y_train"],
        X_val=context["X_val"],
        y_val=context["y_val"],
        alpha=alpha,
    )

    print("\nResultado de poda:")
    print(
        f"  alpha={pruning_results['alpha']:.2f} | "
        f"arcos originales={pruning_results['total_arcs']} | "
        f"arcos activos={pruning_results['active_arcs']} | "
        f"esparsidad={100 * pruning_results['sparsity']:.1f}%"
    )

    test_results = test_model(
        model=pruned_model,
        dataset_name=dataset_name,
        columns_to_drop=context["columns_to_drop"],
        selected_features=context["selected_features"],
        sensor_scaler=context["sensor_scaler"],
        derived_scaler=context["derived_scaler"],
    )

    MODELS_DIR.mkdir(parents=True, exist_ok=True)
    torch.save(
        {
            "dataset": dataset_name,
            "variant": "pruned",
            "architecture": pruned_model.architecture,
            "weights": pruned_model.get_weights(),
            "input_size": pruned_model.input_size,
            "pruning": pruning_results,
        },
        MODELS_DIR / f"{dataset_name}_lightweight_mlp_mopso_pruned.pt",
    )

    return {
        "Dataset": dataset_name,
        "Variant": "pruned",
        "Architecture": processed["Architecture"],
        "Input variables": processed["Input variables"],
        "Parameters": processed["Parameters"],
        "Active arcs": pruning_results["active_arcs"],
        "Total arcs": pruning_results["total_arcs"],
        "Sparsity": pruning_results["sparsity"],
        "Pruning alpha": pruning_results["alpha"],
        "Test MSE": test_results.get("test_mse", np.nan),
        "Test RMSE": test_results.get("test_rmse", np.nan),
        "Test MAE": test_results.get("test_mae", np.nan),
        "Test NASA": test_results.get("test_nasa", np.nan),
        "Test MSE IC95": test_results.get("test_mse_ci95", (np.nan, np.nan)),
        "Test RMSE IC95": test_results.get("test_rmse_ci95", (np.nan, np.nan)),
        "Test MAE IC95": test_results.get("test_mae_ci95", (np.nan, np.nan)),
        "Test NASA IC95": test_results.get("test_nasa_ci95", (np.nan, np.nan)),
        "Time_sec": round(time.time() - pruning_start, 2),
    }

## 4. PODA

$$
\min z = RMSE_{\mathrm{val}}
$$

sujeto a

$$
A_{\mathrm{activo}}
\leq
\left\lfloor
\alpha A_0
\right\rfloor
$$

con

$$
0 < \alpha \leq 1
$$

## Poda de arcos mediante una restricción de esparsidad

Una vez seleccionadas las arquitecturas candidatas de la MLP mediante MOPSO, se introduce una segunda etapa de optimización destinada a reducir el número de conexiones activas de la red.

En esta etapa, la topología a nivel de capas permanece fija. La poda actúa sobre las conexiones individuales de las capas `Linear`, eliminando aquellos pesos que se consideran menos relevantes.

### Definición de arco

Para una MLP con dimensión de entrada $n_0$, capas ocultas de tamaños

$$
(n_1,n_2,\ldots,n_L)
$$

y una salida escalar $n_{L+1}=1$, el número de arcos de la red densa original es

$$
A_0 =
\sum_{\ell=1}^{L+1}
n_{\ell-1}n_{\ell}
$$

Cada elemento de una matriz de pesos $W^{(\ell)}$ representa un arco dirigido entre dos neuronas.

Los términos de sesgo (`bias`) no se consideran arcos. Por tanto, esta medida es diferente de `n_parameters`, utilizada anteriormente en el MOPSO, ya que `n_parameters` incluye tanto pesos como sesgos.

### Variables de poda

Para cada peso $w_{ij}^{(\ell)}$ se define una variable binaria

$$
m_{ij}^{(\ell)} \in {0,1},
$$

donde

$$
m_{ij}^{(\ell)} =
\begin{cases}
1, & \text{si el arco se conserva},\
0, & \text{si el arco se elimina}.
\end{cases}
$$

La matriz efectiva de pesos de cada capa se define entonces como

$$
\widetilde{W}^{(\ell)}
=

W^{(\ell)} \odot M^{(\ell)},
$$

donde $\odot$ representa el producto elemento a elemento.

### Modelo de optimización

Sea $\alpha$ la fracción máxima de arcos de la red original que pueden permanecer activos, con

$$
0 < \alpha \leq 1.
$$

Para un valor de $\alpha$ fijado previamente, el problema de poda puede escribirse como

$$
\min_{W,M} \quad
z =
RMSE_{\mathrm{val}}
\left(
y,
f(X,W\odot M,b)
\right)
$$

sujeto a

$$
\sum_{\ell=1}^{L+1}
\sum_{i=1}^{n_{\ell-1}}
\sum_{j=1}^{n_{\ell}}
m_{ij}^{(\ell)}
\leq
\left\lfloor
\alpha A_0
\right\rfloor,
$$

$$
m_{ij}^{(\ell)}\in{0,1}.
$$

De forma explícita, el objetivo utilizado es

$$
RMSE_{\mathrm{val}}
=

\sqrt{
\frac{1}{N_{\mathrm{val}}}
\sum_{k=1}^{N_{\mathrm{val}}}
\left(
y_k-\hat y_k
\right)^2
}.
$$

Para mantener la misma definición empleada durante la búsqueda MOPSO, las predicciones de validación pueden limitarse al intervalo de RUL:

$$
\hat y_k =
\operatorname{clip}
\left(
f(x_k),0,RUL_{\max}
\right),
\qquad
RUL_{\max}=125.
$$

### Interpretación de $\alpha$

En esta formulación, $\alpha$ representa la **fracción máxima de arcos conservados**, no el número absoluto de arcos.

Por ejemplo,

$$
\alpha=0.70
$$

significa que como máximo puede permanecer activo el $70%$ de los arcos originales, por lo que al menos el $30%$ debe ser eliminado.

Si

$$
A_0=5000,
$$

entonces

$$
A_{\mathrm{activo}}
\leq
\lfloor0.70(5000)\rfloor
=

3500.

$$

La esparsidad obtenida puede expresarse como

$$
S
=

1-
\frac{A_{\mathrm{activo}}}{A_0}.
$$

Por tanto, idealmente,

$$
S \geq 1-\alpha.
$$

### Estrategia práctica de solución

La optimización conjunta de todos los pesos y todas las variables binarias $m_{ij}^{(\ell)}$ constituye un problema combinatorio no convexo de gran dimensión.

Por esta razón, se utiliza una aproximación en dos etapas:

1. Entrenar y seleccionar primero la arquitectura densa mediante el MOPSO existente.
2. Ordenar globalmente los pesos de las capas `Linear` de acuerdo con su magnitud absoluta.
3. Conservar únicamente los

   $$
   K=\lfloor\alpha A_0\rfloor
   $$

   pesos de mayor magnitud.
4. Asignar máscara cero al resto de los pesos.
5. Mantener las máscaras fijas y realizar un proceso corto de *fine-tuning*.
6. Seleccionar el checkpoint con menor RMSE de validación.
7. Evaluar el conjunto de test solamente después de haber fijado tanto la arquitectura como el nivel de poda.

Esta estrategia corresponde a **global unstructured magnitude pruning**, porque la comparación de magnitudes se realiza globalmente entre todas las capas y se eliminan conexiones individuales.

El código de este cuaderno genera una copia podada de cada candidato para cada valor de `PRUNING_ALPHAS`. Estas variantes conservan las activaciones y los biases originales; el fine-tuning y la evaluación de validación/test son etapas posteriores y no se ejecutan en esta celda.

### Relación con MOPSO

La búsqueda MOPSO actual determina

$$
(\text{profundidad},
\text{anchuras},
\text{activación},
\text{dropout})
$$

utilizando como objetivos el RMSE y el número total de parámetros.

La poda se ejecuta posteriormente sobre la solución seleccionada:

$$
\text{MOPSO}
\longrightarrow
\text{MLP densa seleccionada}
\longrightarrow
\text{poda}
\longrightarrow
\text{fine-tuning}
\longrightarrow
\text{modelo final}.
$$

Se evita introducir una variable binaria por cada arco dentro de la codificación del PSO, ya que esto aumentaría la dimensión de búsqueda desde unas pocas variables arquitectónicas hasta potencialmente miles de variables binarias y además interferiría con el mecanismo actual de caché y *warm start*.

### Selección de $\alpha$

Si $\alpha$ se considera una variable de decisión y el único objetivo es minimizar el RMSE, la solución tendería naturalmente hacia

$$
\alpha \rightarrow 1,
$$

es decir, hacia una red sin poda.

Por ello, $\alpha$ debe:

* fijarse como un presupuesto de complejidad; o
* evaluarse sobre un conjunto discreto de valores utilizando únicamente validación; o
* incorporarse como un segundo objetivo.

Una posible extensión multiobjetivo sería

$$
\min
\left(
RMSE_{\mathrm{val}},
\frac{A_{\mathrm{activo}}}{A_0}
\right).
$$

Esto permitiría obtener un nuevo frente de Pareto entre precisión predictiva y esparsidad.

La opción seleccionada con respecto a $\alpha$, al menos por ahora, es la de evaluarse sobre un conjunto discreto de valores.

In [ ]:
def run_pruning_pipeline(
    dense_results: Dict[str, Dict[str, Any]],
    alphas: list = [1, 0.95, 0.90, 0.80, 0.70, 0.60, 0.50],
    datasets: Optional[Iterable[str]] = None,
) -> pd.DataFrame:
    """Poda resultados densos ya calculados, sin repetir process_dataset."""
    selected_datasets = (
        list(dense_results) if datasets is None else list(datasets)
    )
    pruning_summaries: List[Dict[str, Any]] = []

    for dataset_name in selected_datasets:
        if dataset_name not in dense_results:
            raise KeyError(
                f"No hay un resultado denso en memoria para {dataset_name}."
            )
        for alpha in alphas:
            pruning_summaries.append(
                process_pruning(dense_results[dataset_name], alpha=alpha)
            )

    pruning_dataframe = pd.DataFrame(pruning_summaries)
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    pruning_dataframe.to_csv(
        RESULTS_DIR / "mopso_lightweight_pruning_summary.csv",
        index=False,
    )
    print("Resumen de modelos podados:")
    print(pruning_dataframe)
    return pruning_dataframe

pruning_results = run_pruning_pipeline(
    dense_results, alphas=[1, 0.95, 0.90, 0.80, 0.70, 0.60, 0.50], datasets=DATASETS
)

Variantes podadas: 44 (4 candidatos × 11 alphas).
    candidate_id  selected_winner  representative_seed  \
0   candidate_01            False                   44   
1   candidate_01            False                   44   
2   candidate_01            False                   44   
3   candidate_01            False                   44   
4   candidate_01            False                   44   
5   candidate_01            False                   44   
6   candidate_01            False                   44   
7   candidate_01            False                   44   
8   candidate_01            False                   44   
9   candidate_01            False                   44   
10  candidate_01            False                   44   
11  candidate_02            False                   33   
12  candidate_02            False                   33   
13  candidate_02            False                   33   
14  candidate_02            False                   33   
15  candidate_02      